In [ ]:
import numpy as np
from pathlib import Path
from funlib.persistence import open_ds

# Path to your zarr file
zarr_path = "E:/[PROJ]_DCIS/DATA/BRACS/test/raw_and_annotations/Group_AT/Type_ADH/BRACS_1003691.zarr"

# Open the labels at s0 (40x)
labels = open_ds(f"{zarr_path}/labels/s0")

# Load the full mask
mask = labels[:]

print(f"Mask shape: {mask.shape}")
print(f"Mask dtype: {mask.dtype}")
print(f"\nUnique values: {np.unique(mask)}")
print(f"Value counts:")
unique, counts = np.unique(mask, return_counts=True)
for val, count in sorted(zip(unique, counts)):
    pct = 100 * count / mask.size
    print(f"  {val}: {count:,} pixels ({pct:.2f}%)")

# Check if any non-255 values exist
non_ignore = mask[mask != 255]
if len(non_ignore) > 0:
    print(f"\n✓ Found {len(non_ignore):,} annotated pixels!")
    print(f"  Annotation classes: {np.unique(non_ignore)}")
else:
    print(f"\n✗ All pixels are 255 (IGNORE) — no annotations found!")

# Try other levels too
for level in [1, 2, 3]:
    try:
        labels_level = open_ds(f"{zarr_path}/labels/s{level}")
        mask_level = labels_level[:]
        non_ignore_level = mask_level[mask_level != 255]
        print(f"\ns{level}: {len(non_ignore_level):,} annotated pixels")
    except:
        print(f"\ns{level}: Not found or error")


In [ ]:
import numpy as np
import napari
from pathlib import Path

# Load a single NPZ tile
npz_path = r"E:\[PROJ]_DCIS\DATA\BRACS\test\training_patches_40x\Group_MT\Type_DCIS\BRACS_1945\BRACS_1945_tile_y7680600_x26324127.npz"  # ← Change this
data = np.load(npz_path)

image = data['image']
label = data['label']

print(f"Image shape: {image.shape}, dtype: {image.dtype}")
print(f"Label shape: {label.shape}, dtype: {label.dtype}")
print(f"Unique labels: {np.unique(label)}")

# Create viewer and add layers
viewer = napari.Viewer()

# Add image layer (RGB)
viewer.add_image(image, name='H&E', rgb=True)

# Add label layer (with random colors)
viewer.add_labels(label, name='Annotations', opacity=0.5)

# Optional: add as image layer instead (to see exact pixel values)
# viewer.add_image(label, name='Annotations (grayscale)', colormap='viridis', opacity=0.5)

napari.run()


In [ ]:
# ============================================================================
# Cell 1: Import libraries
# ============================================================================
import os
import numpy as np
import pandas as pd
import json
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm
import matplotlib.pyplot as plt

# ============================================================================
# Cell 2: Define parameters
# ============================================================================
# UPDATE THESE PARAMETERS
npz_folder = r"E:\[PROJ]_DCIS\DATA\BRACS\test\training_patches_40x\Group_AT\Type_ADH\BRACS_1844"  # Change this to your folder path
ignore_label = 255  # Label value to ignore (typically background/padding)
save_json = False  # Whether to save results to JSON
output_json_path = None  # Leave None to auto-generate, or specify a path

# ============================================================================
# Cell 3: Define functions
# ============================================================================
def count_pixels_per_label(npz_folder, ignore_label=255, verbose=True):
    """
    Count pixel occurrences per label across all .npz files.
    
    Args:
        npz_folder (str): Path to folder containing .npz files
        ignore_label (int): Label value to ignore in counts (default: 255)
        verbose (bool): Print progress bar
    
    Returns:
        dict: {label: pixel_count}
    """
    npz_folder = Path(npz_folder)
    npz_files = sorted(npz_folder.glob("*.npz"))
    
    if not npz_files:
        print(f"No .npz files found in {npz_folder}")
        return {}
    
    print(f"Found {len(npz_files)} .npz files")
    
    label_counts = defaultdict(int)
    
    iterator = tqdm(npz_files, desc="Processing files") if verbose else npz_files
    
    for npz_file in iterator:
        try:
            data = np.load(npz_file)
            
            if "label" not in data:
                print(f"Warning: {npz_file.name} has no 'label' key, skipping")
                continue
            
            label_array = data["label"]
            
            # Count pixels per label (excluding ignore_label)
            unique, counts = np.unique(label_array, return_counts=True)
            
            for label, count in zip(unique, counts):
                if label != ignore_label:
                    label_counts[label] += count
                    
        except Exception as e:
            print(f"Error reading {npz_file.name}: {e}")
            continue
    
    return dict(sorted(label_counts.items()))


def print_label_statistics(label_counts, ignore_label=255, title="Label Statistics"):
    """
    Pretty-print label statistics as a formatted table.
    
    Args:
        label_counts (dict): {label: pixel_count}
        ignore_label (int): Label value being ignored
        title (str): Title to print
    """
    if not label_counts:
        print("No labels found!")
        return
    
    total_pixels = sum(label_counts.values())
    
    print("\n" + "="*70)
    print(f"{title}")
    print("="*70)
    print(f"Ignore label: {ignore_label}")
    print(f"Total valid pixels: {total_pixels:,}")
    print(f"Number of unique labels: {len(label_counts)}")
    print("-"*70)
    print(f"{'Label':<10} {'Pixel Count':<20} {'Percentage':<15}")
    print("-"*70)
    
    for label, count in sorted(label_counts.items()):
        pct = (count / total_pixels * 100) if total_pixels > 0 else 0
        print(f"{label:<10} {count:<20,} {pct:>6.2f}%")
    
    print("="*70 + "\n")


def save_label_statistics(label_counts, output_path):
    """
    Save label statistics to JSON file.
    
    Args:
        label_counts (dict): {label: pixel_count}
        output_path (str): Path to save JSON file
    """
    # Convert to string keys for JSON compatibility
    stats = {
        "total_pixels": sum(label_counts.values()),
        "num_labels": len(label_counts),
        "label_counts": {str(k): v for k, v in label_counts.items()},
        "label_percentages": {
            str(k): round(v / sum(label_counts.values()) * 100, 2) 
            for k, v in label_counts.items()
        }
    }
    
    with open(output_path, "w") as f:
        json.dump(stats, f, indent=2)
    
    print(f"✓ Statistics saved to {output_path}")


# ============================================================================
# Cell 4: Run analysis
# ============================================================================
# Validate folder exists
if not Path(npz_folder).exists():
    print(f"Error: Folder not found: {npz_folder}")
else:
    # Count pixels per label
    label_counts = count_pixels_per_label(
        npz_folder, 
        ignore_label=ignore_label, 
        verbose=True
    )
    
    # Print statistics
    print_label_statistics(label_counts, ignore_label=ignore_label)
    
    # Save to JSON if requested
    if save_json and label_counts:
        if output_json_path is None:
            output_json_path = Path(npz_folder).parent / f"{Path(npz_folder).name}_label_stats.json"
        save_label_statistics(label_counts, output_json_path)

# ============================================================================
# Cell 5: Visualize (optional)
# ============================================================================
if label_counts:
    # Create DataFrame for easier manipulation
    df_stats = pd.DataFrame({
        "Label": list(label_counts.keys()),
        "Pixel Count": list(label_counts.values())
    })
    df_stats["Percentage"] = (df_stats["Pixel Count"] / df_stats["Pixel Count"].sum() * 100)
    
    print("\nDataFrame Summary:")
    print(df_stats.to_string(index=False))
    
    # Plot bar chart
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Absolute counts
    ax1.bar(df_stats["Label"].astype(str), df_stats["Pixel Count"], color="steelblue")
    ax1.set_xlabel("Label")
    ax1.set_ylabel("Pixel Count")
    ax1.set_title("Pixel Count per Label")
    ax1.grid(axis="y", alpha=0.3)
    
    # Percentage
    colors = plt.cm.Set3(np.linspace(0, 1, len(df_stats)))
    ax2.pie(df_stats["Pixel Count"], labels=df_stats["Label"].astype(str), 
            autopct="%1.1f%%", colors=colors, startangle=90)
    ax2.set_title("Label Distribution")
    
    plt.tight_layout()
    plt.show()


In [4]:
import numpy as np
from pathlib import Path
from collections import Counter
import csv
from datetime import datetime

# === CONFIGURE HERE ===
ROOT_DIR = r"E:\[PROJ]_DCIS\DATA\BRACS\test\training_patches"
BATCH_LEVEL = 3  # 3 = process 3 levels deep (e.g., 40x/Group_MT/Type_IC)
OUTPUT_CSV = r"E:\[PROJ]_DCIS\DATA\BRACS\test\training_patches\label_counts.csv"
# ======================

def get_batch_folders(root_dir, depth):
    """
    Get all folders at a specific depth level.
    depth=1: immediate subfolders
    depth=2: subfolders of subfolders, etc.
    """
    root_path = Path(root_dir)
    folders = [root_path]
    
    for _ in range(depth):
        new_folders = []
        for folder in folders:
            new_folders.extend(folder.iterdir())
        folders = [f for f in new_folders if f.is_dir()]
    
    return sorted(folders)


def count_labels_in_folder(folder_path):
    """Count labels in a single folder's .npz files. Returns Counter instead of list."""
    folder_path = Path(folder_path)
    npz_files = list(folder_path.rglob("*.npz"))
    
    if not npz_files:
        return None, []
    
    batch_counter = Counter()
    failed = []
    
    for npz_file in npz_files:
        try:
            data = np.load(npz_file, allow_pickle=False)
            if 'label' not in data:
                failed.append((npz_file.name, f"Missing 'label' key. Has: {list(data.keys())}"))
                data.close()
                continue
            
            # Count labels for this file and add to batch counter
            labels = data['label'].flatten()
            batch_counter.update(labels)
            data.close()
        except Exception as e:
            failed.append((npz_file.name, f"{type(e).__name__}: {e}"))
    
    return batch_counter if batch_counter else None, failed


def count_unique_labels_by_batch(root_dir, batch_level=2, output_csv=None):
    """Count unique labels by processing folders in batches, then aggregate and save."""
    root_path = Path(root_dir)
    
    print(f"Counting labels by batch (depth={batch_level})...\n")
    
    batch_folders = get_batch_folders(root_dir, batch_level)
    
    if not batch_folders:
        print("No folders found at specified depth")
        return
    
    print(f"Found {len(batch_folders)} batch folders to process\n")
    
    aggregated_counts = Counter()  # Running total across all batches
    total_failed = []
    batch_results = []  # Store per-batch results for CSV
    
    for i, folder in enumerate(batch_folders, 1):
        folder_name = "/".join(folder.relative_to(root_path).parts)
        folder_parts = list(folder.relative_to(root_path).parts)
        
        print(f"[{i}/{len(batch_folders)}] Processing: {folder_name}")
        
        batch_counter, failed = count_labels_in_folder(folder)
        
        if batch_counter is None:
            print(f"  → No .npz files found")
            continue
        
        # Add this batch's counts to the aggregated total
        aggregated_counts.update(batch_counter)
        total_failed.extend([(folder_name, f) for f in failed])
        
        # Calculate total pixels in this batch
        total_pixels_in_batch = sum(batch_counter.values())
        
        # Store for CSV output
        for label_val in sorted(batch_counter.keys()):
            count = batch_counter[label_val]
            pct = 100 * count / total_pixels_in_batch
            batch_results.append({
                'folder_parts': folder_parts,
                'label': label_val,
                'count': count,
                'percentage': pct,
                'total_pixels': total_pixels_in_batch
            })
        
        num_files = len(list(folder.rglob('*.npz')))
        print(f"  ✓ Processed {total_pixels_in_batch:,} labels from {num_files} files")
        
        if failed:
            print(f"  ⚠ {len(failed)} files failed")
        print()
    
    if not aggregated_counts:
        print("No valid labels found!")
        return
    
    unique_labels = sorted(aggregated_counts.keys())
    total_pixels = sum(aggregated_counts.values())
    
    print("\n" + "="*60)
    print("FINAL RESULTS - Unique label values and frequencies (ALL):")
    print("-" * 60)
    for label_val in unique_labels:
        count = aggregated_counts[label_val]
        pct = 100 * count / total_pixels
        print(f" Label {label_val:3d}: {count:12,} pixels ({pct:6.2f}%)")
    
    print("-" * 60)
    print(f"Total unique labels: {len(unique_labels)}")
    print(f"Total pixels: {total_pixels:,}")
    print(f"Batches processed: {len(batch_folders)}")
    print(f"Total files failed: {len(total_failed)}")
    
    if total_failed:
        print("\n" + "="*60)
        print("FAILED FILES SUMMARY:")
        print("-" * 60)
        for batch_name, (fname, reason) in total_failed[:20]:
            print(f"{batch_name}/{fname}")
            print(f"  → {reason}\n")
        if len(total_failed) > 20:
            print(f"... and {len(total_failed) - 20} more\n")
    
    # Save to CSV with hierarchical folder structure
    if output_csv:
        output_csv = Path(output_csv)
        output_csv.parent.mkdir(parents=True, exist_ok=True)
        
        with open(output_csv, 'w', newline='') as csvfile:
            # Determine max folder depth from batch results
            max_depth = max(len(r['folder_parts']) for r in batch_results) if batch_results else 0
            
            # Create column headers for folder hierarchy
            folder_cols = [f'Level_{i+1}' for i in range(max_depth)]
            fieldnames = folder_cols + ['Label', 'Count', 'Percentage', 'Total_Pixels_In_Batch']
            
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()
            
            for result in batch_results:
                row = {}
                # Fill in folder hierarchy
                for i, part in enumerate(result['folder_parts']):
                    row[f'Level_{i+1}'] = part
                # Fill remaining level columns with empty strings
                for i in range(len(result['folder_parts']), max_depth):
                    row[f'Level_{i+1}'] = ''
                # Fill in data columns
                row['Label'] = result['label']
                row['Count'] = result['count']
                row['Percentage'] = f"{result['percentage']:.2f}%"
                row['Total_Pixels_In_Batch'] = result['total_pixels']
                
                writer.writerow(row)
        
        print(f"\n✓ Results saved to: {output_csv}")
    
    return aggregated_counts


count_unique_labels_by_batch(ROOT_DIR, BATCH_LEVEL, OUTPUT_CSV)

Counting labels by batch (depth=3)...

Found 14 batch folders to process

[1/14] Processing: 20x/Group_AT/Type_ADH
  ✓ Processed 168,992,768 labels from 3368 files

[2/14] Processing: 20x/Group_AT/Type_FEA
  ✓ Processed 292,024,320 labels from 5820 files

[3/14] Processing: 20x/Group_BT/Type_N
  ✓ Processed 19,117,056 labels from 381 files

[4/14] Processing: 20x/Group_BT/Type_PB
  ✓ Processed 26,994,688 labels from 538 files

[5/14] Processing: 20x/Group_BT/Type_UDH
  ✓ Processed 36,076,544 labels from 719 files

[6/14] Processing: 20x/Group_MT/Type_DCIS
  ✓ Processed 115,404,800 labels from 2300 files

[7/14] Processing: 20x/Group_MT/Type_IC
  ✓ Processed 24,536,064 labels from 489 files

[8/14] Processing: 40x/Group_AT/Type_ADH
  ✓ Processed 511,092,736 labels from 10186 files

[9/14] Processing: 40x/Group_AT/Type_FEA
  ✓ Processed 770,151,424 labels from 15349 files

[10/14] Processing: 40x/Group_BT/Type_N
  ✓ Processed 33,818,624 labels from 674 files

[11/14] Processing: 40x/Grou

Counter({255: 1595863896,
         2: 1497501715,
         3: 1107307369,
         0: 681128913,
         4: 312064693,
         1: 241097382})

In [1]:
import numpy as np
from pathlib import Path
import zipfile
import traceback
import os
from datetime import datetime

# === CONFIGURE HERE ===
npz_file_path = r"E:\[PROJ]_DCIS\DATA\BRACS\test\training_patches\40x\Group_MT\Type_IC\BRACS_286\BRACS_286_tile_y9305100_x15619464.npz"
# ======================

def diagnose_npz(npz_path):
    """Diagnose why an .npz file is corrupted."""
    npz_path = Path(npz_path)
    
    print(f"Diagnosing: {npz_path.name}")
    print("=" * 70)
    
    # 1. Check file exists and size
    if not npz_path.exists():
        print(f"❌ File does not exist")
        return
    
    file_size = npz_path.stat().st_size
    print(f"✓ File exists")
    print(f"  Size: {file_size:,} bytes ({file_size/1024:.2f} KB)")
    
    if file_size == 0:
        print(f"❌ File is EMPTY (0 bytes) - likely never written")
        return
    
    # 2. Check if it's a valid ZIP format (npz is just a zip)
    print(f"\n--- Checking ZIP structure ---")
    try:
        with zipfile.ZipFile(npz_path, 'r') as zf:
            file_list = zf.namelist()
            print(f"✓ Valid ZIP file")
            print(f"  Contains {len(file_list)} files:")
            for fname in file_list:
                finfo = zf.getinfo(fname)
                print(f"    - {fname} ({finfo.file_size:,} bytes)")
    except zipfile.BadZipFile as e:
        print(f"❌ CORRUPT ZIP: {e}")
        print(f"   File is truncated or not a valid ZIP archive")
        print(f"   This means: the .npz was never fully written")
        return
    except Exception as e:
        print(f"❌ ZIP Error: {e}")
        return
    
    # 3. Try to load with numpy
    print(f"\n--- Loading with numpy.load() ---")
    try:
        data = np.load(npz_path, allow_pickle=False)
        print(f"✓ Loaded successfully")
        print(f"  Keys: {list(data.keys())}")
        for key in data.keys():
            arr = data[key]
            print(f"    - '{key}': shape={arr.shape}, dtype={arr.dtype}")
        data.close()
    except Exception as e:
        print(f"❌ LOAD ERROR: {type(e).__name__}")
        print(f"   Message: {e}")
        traceback.print_exc()
        print(f"\n   Likely cause: Corrupted ZIP (incomplete write)")
    
    # 4. Check file timestamp
    print(f"\n--- File metadata ---")
    stat = npz_path.stat()
    mod_time = datetime.fromtimestamp(stat.st_mtime)
    print(f"  Modified: {mod_time}")
    print(f"  Readable: {os.access(npz_path, os.R_OK)}")

# Run diagnosis
diagnose_npz(npz_file_path)


Diagnosing: BRACS_286_tile_y9305100_x15619464.npz
✓ File exists
  Size: 652,794 bytes (637.49 KB)

--- Checking ZIP structure ---
✓ Valid ZIP file
  Contains 2 files:
    - image.npy (602,240 bytes)
    - label.npy (50,304 bytes)

--- Loading with numpy.load() ---
✓ Loaded successfully
  Keys: ['image', 'label']
    - 'image': shape=(224, 224, 3), dtype=float32
    - 'label': shape=(224, 224), dtype=uint8

--- File metadata ---
  Modified: 2026-09-16 03:28:12.355866
  Readable: True
